# 04 — Parcels: who owns the high ground (Virginia / VGIN)Same job as the NC version, different source. North Carolina's OneMap becomes**VGIN**, the Virginia Geographic Information Network's statewide parcel layer —localities push their parcels to VGIN, which aggregates them quarterly. Free,no key, same ArcGIS query pattern, so the fetch logic is a near-straight port.```https://vginmaps.vdem.virginia.gov/arcgis/rest/services/VA_Base_Layers/VA_Parcels/MapServer/0```Two Virginia-specific things to know:- **Attributes vary by locality.** VGIN standardizes geometry and a core schema,  but not every locality shares owner names, and field names drift. The probe  cell below reads the *live* field list so we map real columns, not remembered  ones. Display field is `PARCELID`; everything else we detect.- **1000-record page cap** (lower than NC's 5000), so paging matters more. The  fetch loop handles it.As in NC: a parcel says who owns land and where its edges are. It does **not**say what's for sale — that's notebook 05, a separate world. No "for sale"column exists here.

## 1 · Config

In [ ]:
from pathlib import Pathimport timeimport numpy as np, pandas as pdimport requestsimport geopandas as gpdimport va_aoi as AOUT_DIR = Path("output")CACHE   = Path("data/parcels"); CACHE.mkdir(parents=True, exist_ok=True)VGIN = ("https://vginmaps.vdem.virginia.gov/arcgis/rest/services/"        "VA_Base_Layers/VA_Parcels/MapServer/0")MIN_ACRES, MAX_ACRES = A.MIN_SITE_ACRES, A.MAX_SITE_ACRESPAGE = 1000                     # VGIN layer cap

## 2 · Probe the service

In [ ]:
# --- Confirm the service and read its REAL fields ---------------------------meta = requests.get(VGIN, params={"f": "json"}, timeout=60).json()print(meta.get("name"), "|", meta.get("geometryType"))print("max records:", meta.get("maxRecordCount"))print("\nfields:")for f in meta.get("fields", []):    print(f"  {f['name']:<22} {f['type'].replace('esriFieldType','')}")

## 3 · Paged fetch

In [ ]:
def fetch_parcels(bbox_wgs84, page=PAGE):    """Page an ArcGIS layer for everything intersecting a bbox, in EPSG:4326."""    minx, miny, maxx, maxy = bbox_wgs84    feats, offset = [], 0    while True:        r = requests.get(f"{VGIN}/query", timeout=180, params={            "f": "geojson", "where": "1=1", "outFields": "*",            "geometry": f"{minx},{miny},{maxx},{maxy}",            "geometryType": "esriGeometryEnvelope",            "inSR": 4326, "outSR": 4326,            "spatialRel": "esriSpatialRelIntersects",            "resultOffset": offset, "resultRecordCount": page,            "returnGeometry": "true",        })        r.raise_for_status()        got = r.json().get("features", [])        feats.extend(got)        print(f"    +{len(got):>4}  (total {len(feats):,})")        if len(got) < page:            break        offset += page        time.sleep(0.4)        if offset > 120_000:            print("    [warn] bailing at 120k - tighten bbox"); break    return (gpd.GeoDataFrame.from_features(feats, crs=4326)            if feats else gpd.GeoDataFrame(geometry=[], crs=4326))

## 4 · Pull per site

In [ ]:
# --- Pull parcels around each surviving site --------------------------------sites = pd.read_csv(OUT_DIR / "site_checklist.csv")if sites.get("keep", pd.Series(dtype=str)).astype(str).str.lower().eq("y").any():    sites = sites[sites.keep.astype(str).str.lower() == "y"]    print(f"using {len(sites)} sites marked keep=y")else:    sites = sites[sites.exclusion.fillna("") == ""]    print(f"no keep column filled - using all {len(sites)} unflagged sites")PAD_DEG = 0.02frames = []for _, s in sites.iterrows():    bbox = (s.lon-PAD_DEG, s.lat-PAD_DEG, s.lon+PAD_DEG, s.lat+PAD_DEG)    key = CACHE / f"{s.locality}_{s.lat:.4f}_{s.lon:.4f}.gpkg"    if key.exists():        g = gpd.read_file(key); print(f"  cached  {s.locality} {s.lat:.4f}  {len(g):,}")    else:        print(f"  fetch   {s.locality} {s.lat:.4f},{s.lon:.4f}")        g = fetch_parcels(bbox)        if len(g): g.to_file(key, driver="GPKG")    if len(g):        g["site_beach_mi"] = s.beach_mi        g["site_mean_ft"] = s.mean_ft        frames.append(g)parcels = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=4326) \          if frames else gpd.GeoDataFrame(geometry=[], crs=4326)print(f"\n{len(parcels):,} parcel records")

## 5 · Normalize + filter

In [ ]:
# --- Normalize (fields detected from the probe) -----------------------------def pick(df, *names, default=None):    for n in names:        if n in df.columns: return df[n]        for c in df.columns:               # case-insensitive fallback            if c.lower() == n.lower(): return df[c]    return pd.Series([default]*len(df), index=df.index)p = parcels.drop_duplicates(subset="geometry").to_crs(A.WORKING_EPSG).copy()p["calc_acres"] = p.geometry.area / 4046.8564224p["pid"]      = pick(p, "PARCELID", "GPIN", "PIN", default="")p["owner"]    = pick(p, "OWNERNAME", "OWNER", "OWNER_NAME", "LASTNAME", default="")p["addr"]     = pick(p, "SITADDRES", "SITEADDRESS", "ADDRESS", "PROPADDR", default="")p["locality"] = pick(p, "LOCALITY", "JURIS", "LOCALITY_NAME", "FIPS", default="")p["assessed"] = pd.to_numeric(pick(p, "ASSESSED", "TOTALVALUE", "FMV",                                   default=np.nan), errors="coerce")p["acres"]    = p["calc_acres"]            # trust geometry over statedcand = p[p.acres.between(MIN_ACRES, MAX_ACRES)].copy()cent = cand.geometry.centroid.to_crs(4326)cand["lat"], cand["lon"] = cent.y.values, cent.x.valuescand["maps_url"] = cand.apply(    lambda r: f"https://www.google.com/maps/@{r.lat:.6f},{r.lon:.6f},800m/data=!3m1!1e3", axis=1)print(f"{len(p):,} unique parcels -> {len(cand):,} in the "      f"{MIN_ACRES:g}-{MAX_ACRES:g} acre window")print(cand[["pid","owner","acres","assessed"]].head(12).to_string(index=False))

## 6 · Flag public, save

In [ ]:
# --- Flag public owners, save -----------------------------------------------PUBLIC = ["UNITED STATES","USA","STATE OF","COMMONWEALTH","COUNTY OF","CITY OF",          "TOWN OF","DEPT","DEPARTMENT","COMMISSION","AUTHORITY","BOARD OF",          "MUNICIPAL","NAVY","MARINE","MILITARY","FISH AND WILDLIFE","REFUGE",          "CONSERVAN","TRUST FOR","CHURCH","VIRGINIA"]u = cand.owner.fillna("").str.upper()cand["likely_public"] = u.apply(lambda s: any(h in s for h in PUBLIC))private = cand[~cand.likely_public].copy()print(f"{cand.likely_public.sum()} likely public, {len(private)} private")cols = ["pid","owner","addr","acres","assessed","site_mean_ft",        "site_beach_mi","lat","lon","maps_url"]cols = [c for c in cols if c in private.columns]private[cols].sort_values("acres", ascending=False) \             .to_csv(OUT_DIR / "parcel_candidates.csv", index=False)private.to_file(OUT_DIR / "parcel_candidates.gpkg", driver="GPKG")print(f"[ok] output/parcel_candidates.csv  ({len(private)} rows)")print(f"[ok] output/parcel_candidates.gpkg  (QGIS)")